# Agent 数据输入方式全览

LangGraph Agent 有多种向图传递数据的方式，各有不同的生命周期、持久化策略和读取入口。本笔记系统整理所有方式，便于在设计 Agent 时做出正确选择。

## 总览对比

| 方式 | 传入位置 | 持久化 | 跨 invoke 共享 | 读取入口 | 典型用途 |
|------|---------|--------|--------------|---------|--------|
| `messages` | `invoke({"messages": [...]})` | 是（checkpointer） | 是 | `state["messages"]` | 对话历史 |
| `state_schema` 扩展字段 | `invoke({"my_field": ...})` | 是（checkpointer） | 是 | `state["my_field"]` / `runtime.state` | 业务数据 |
| `context_schema` | `invoke(..., context=MyCtx(...))` | 否 | 否 | `runtime.context.field` | 运行时配置 |
| `store` | `create_agent(store=store)` | 是（外部存储） | 跨 thread | `runtime.store.get(...)` | 长期记忆 |
| `config` | `invoke(..., config={...})` | 否 | 否 | `config["configurable"]["key"]` | thread_id 等元信息 |
| `Command` | `invoke(Command(resume=...))` | — | — | 中断恢复专用 | HITL 审批 |

## 1. `messages`：基础对话输入

最基本的输入方式，所有 Agent 都必须有。

```python
agent.invoke(
    {"messages": [HumanMessage(content="你好")]},
    config={"configurable": {"thread_id": "1"}}
)
```

- 底层由 `AgentState` 定义：`messages: Required[Annotated[list[AnyMessage], add_messages]]`
- `add_messages` 是 reducer，新消息**追加**而非覆盖
- 有 checkpointer 时，消息历史在同一 `thread_id` 下跨 invoke 累积

## 2. `state_schema`：扩展持久化状态

继承 `AgentState` 添加自定义字段，字段值随 state 一起被 checkpointer 持久化。

### 定义与注册

```python
from langchain.agents import AgentState, create_agent

class EmailState(AgentState):
    email: str   # 新增字段

agent = create_agent(
    model="gpt-4o-mini",
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
)
```

### 传入与读取

```python
# invoke 时直接在 input dict 里写字段名
agent.invoke(
    {
        "messages": [HumanMessage(...)],
        "email": "Hi Seán..."   # ← 直接写，LangGraph 按 schema 合并进 state
    },
    config=config
)

# 中间件中读取（runtime.state）
@before_agent
def my_middleware(state: EmailState, runtime: Runtime):
    email = state["email"]

# 工具中读取（ToolRuntime.state）
@tool
def read_email(runtime: ToolRuntime) -> str:
    return runtime.state["email"]
```

### 底层机制（`factory.py:968`）

```python
# create_agent 内部合并所有 schema（包括中间件的）
base_state = state_schema if state_schema is not None else AgentState
resolved_state_schema = _resolve_schema({base_state, *middleware_schemas})
graph = StateGraph(state_schema=resolved_state_schema)
```

**适用场景：** 业务数据（邮件内容、用户信息、中途积累的分析结果），需要在中断恢复后依然存在的数据。

## 3. `context_schema`：运行时只读配置

每次 `invoke` 独立传入，**不持久化**，不参与 state 流转，适合描述「这次执行的环境」。

### 定义与注册

```python
from dataclasses import dataclass

@dataclass
class LanguageContext:
    user_language: str = "English"

agent = create_agent(
    model="gpt-4o-mini",
    context_schema=LanguageContext,
    middleware=[user_language_prompt],
)
```

### 传入与读取

```python
# invoke 时通过 context= 参数传入（不在 input dict 里）
agent.invoke(
    {"messages": [HumanMessage(...)]},
    context=LanguageContext(user_language="Irish")   # ← 独立参数
)

# 中间件中读取（runtime.context）
@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    lang = request.runtime.context.user_language

# 工具中读取（ToolRuntime.context）
@tool
def my_tool(runtime: ToolRuntime) -> str:
    return runtime.context.user_language
```

### 与 state_schema 的关键区别

```
state_schema → invoke({"email": ...})   → 写入 state → checkpointer 存档 → 中断恢复后还在
context_schema → invoke(context=...)   → 只在本次执行可见 → 中断恢复时必须重新传
```

**适用场景：** 语言偏好、用户角色/权限、AB 测试分组、数据库连接对象——每次调用可能不同、不需要持久化的配置。

## 4. `store`：跨 thread 长期记忆

与 checkpointer 不同，`store` 是**跨 thread** 的持久化存储，用于在不同会话间共享信息（如用户的长期偏好、知识库）。

### 注册与使用

```python
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(("users",), "user_123", {"name": "Alice", "language": "French"})

agent = create_agent(
    model="gpt-4o-mini",
    store=store,           # ← 编译时注入，所有 thread 共享同一个 store
    checkpointer=InMemorySaver(),
)

# 中间件/工具中读取
@before_agent
def load_user_prefs(state, runtime: Runtime):
    if runtime.store:
        memory = runtime.store.get(("users",), "user_123")
        # 将用户偏好注入 state...
```

### checkpointer vs store

| | checkpointer | store |
|--|--|--|
| **范围** | 单个 thread（会话） | 全局（跨 thread） |
| **数据** | 完整 state 快照 | 任意键值对 |
| **访问** | 框架自动管理 | `runtime.store.get/put` |
| **典型用途** | 中断恢复、会话记忆 | 用户长期偏好、知识库、跨会话记忆 |

**适用场景：** 用户画像、跨会话学习到的偏好、共享知识库。

## 5. `config`：元信息与线程控制

通过 `RunnableConfig` 传入框架级别的元信息，不进入 state，也不由 LLM 感知。

```python
config = {
    "configurable": {
        "thread_id": "user_42_session_1",  # checkpointer 用于区分会话
    },
    "run_name": "email-agent",             # LangSmith trace 名称
    "tags": ["production"],               # LangSmith 标签
}

agent.invoke({"messages": [...]}, config=config)
```

在节点或中间件中读取：

```python
from langchain_core.runnables import RunnableConfig

def my_node(state, config: RunnableConfig):
    thread_id = config["configurable"]["thread_id"]
```

**注意：** `Runtime` 不包含 `config`，需要在函数签名中单独声明 `config: RunnableConfig` 参数才能注入。

**适用场景：** thread 标识、LangSmith 追踪配置、自定义 configurable 参数。

## 6. `Command`：中断恢复专用输入

`Command` 不是普通数据输入，而是向暂停的图发送控制信号，用于 HITL 场景。

```python
from langgraph.types import Command

# 审批通过
agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)

# 拒绝
agent.invoke(Command(resume={"decisions": [{"type": "reject", "message": "..."}]}), config=config)

# 编辑后执行
agent.invoke(Command(resume={"decisions": [{"type": "edit", "edited_action": {...}}]}), config=config)
```

- 必须使用与第一次 invoke **相同的 `thread_id`**，LangGraph 才能找到对应的暂停点
- 图从中断节点继续，而非重新开始
- LLM 对中断完全无感知，看到的是连贯的消息历史

**适用场景：** 人工审批工具调用、编辑 LLM 输出后继续执行。

## 7. `Runtime` 与 `ToolRuntime`：注入对象汇总

上述所有数据最终通过 `Runtime`（中间件/节点）或 `ToolRuntime`（工具）注入到执行函数中：

```
Runtime（注入到中间件和节点）
├── .context    ← context_schema 传入的值
├── .store      ← create_agent(store=...) 传入的存储
├── .stream_writer  ← 流式输出写入器
└── .previous   ← 上一次该 thread 的返回值（functional API）

ToolRuntime（注入到工具，是 Runtime 的超集）
├── .context    ← 同上
├── .store      ← 同上
├── .state      ← 当前完整 state（含 state_schema 扩展字段）
├── .tool_call_id  ← 当前工具调用的 ID
└── .config     ← RunnableConfig（含 thread_id 等）
```

## 选择决策树

```
这份数据的特征是？
│
├── 是对话消息
│     └── messages（必选）
│
├── 业务数据，需要持久化 / 中断恢复后还要用
│     └── state_schema 扩展字段
│
├── 本次调用的环境配置，每次 invoke 可能不同
│     └── context_schema
│
├── 跨会话（thread）共享的长期记忆
│     └── store
│
├── 框架元信息（thread_id、trace 配置等）
│     └── config
│
└── 向暂停的图发送人工审批信号
      └── Command
```